## Week7 Day5: Fine-Tuning an OpenSource Model using QLoRA

In [ ]:
%pip install -qU peft trl bitsandbytes datasets wandb

In [ ]:
import os
import torch
import wandb
import torch.nn.functional as F
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from datetime import datetime
from dotenv import load_dotenv
from evaluate import Tester

In [ ]:
# Load environment variables
load_dotenv(override=True)
hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    raise ValueError("HF_TOKEN is not set")
print("HF_TOKEN is set")
login(hf_token, add_to_git_credential=True)

In [ ]:
# Set the project name
PROJECT_NAME = "home-goods-price-predictor"
BASE_MODEL = "meta-llama/Llama-3.2-1B"
HF_USER = "ebenhays"
DATASET_NAME = "ed-donner/home-data"
RUN_NAME = f"{datetime.now():%Y-%m-%d}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"
REVISION = None

In [ ]:
# Hyperparameters for QLoRA
LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.1
QUANT_4_BIT = True

# Hyperparameters for Training
EPOCHS = 1 
BATCH_SIZE = 4 
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO = 0.01
OPTIMIZER = "paged_adamw_32bit"

# Tracking
LOG_STEPS = 5
SAVE_STEPS = 100 
LOG_TO_WANDB = True

In [ ]:
# Log in to Weights & Biases
wandb_api_key = os.getenv("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
test = train.select(range(1000))

In [ ]:
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16
    )

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL,trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
# Set up the Training parameters
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=False,
)

In [ ]:
# Initalize the Supervised Fine Tuning Trainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train, #train takes a bit of time, you can use test for quick testing 
    peft_config=lora_parameters,
    args=train_parameters,
)

In [ ]:
# Now, let's Fine-tune
fine_tuning.train()

# Save the fine-tuned adapters and tokenizer locally, then push to the Hub
try:
    fine_tuning.model.save_pretrained(PROJECT_RUN_NAME)
    tokenizer.save_pretrained(PROJECT_RUN_NAME)
    # Push adapters + tokenizer to the Hugging Face Hub
    fine_tuning.model.push_to_hub(HUB_MODEL_NAME, token=hf_token)
    tokenizer.push_to_hub(HUB_MODEL_NAME, token=hf_token)
    print(f"Saved local checkpoint and pushed to hub: {HUB_MODEL_NAME}")
except Exception as e:
    print(f"Warning: failed to push to hub: {e}")

In [ ]:
if LOG_TO_WANDB:
    wandb.finish()

In [ ]:
# Load from local
fine_tuned_model = PeftModel.from_pretrained(base_model, PROJECT_RUN_NAME)
print(
    f"Fine-tuned model loaded from local checkpoint. Memory: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB"
)

In [ ]:
def predict_price_from_finetuned_model(prompt, device="cuda"):
    set_seed(42)
    top_K = 5
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
    attention_mask = torch.ones(inputs.shape, device=device)

    with torch.no_grad():
        outputs = fine_tuned_model(inputs, attention_mask=attention_mask)
        next_token_logits = outputs.logits[:, -1, :].to("cpu")

    next_token_probs = F.softmax(next_token_logits, dim=-1)
    top_prob, top_token_id = next_token_probs.topk(top_K)
    prices, weights = [], []
    for i in range(top_K):
        predicted_token = tokenizer.decode(top_token_id[0][i])
        probability = top_prob[0][i]
        try:
            result = float(predicted_token)
        except ValueError as e:
            result = 0.0
        if result > 0:
            prices.append(result)
            weights.append(probability)
    if not prices:
        return 0.0
    total = sum(weights)
    weighted_prices = [price * weight / total for price, weight in zip(prices, weights)]
    return sum(weighted_prices).item()


fine_tuned_model = fine_tuned_model.float()

In [ ]:
Tester.test(predict_price_from_finetuned_model, test)